In [28]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

in my terminal i examined the file using the command: zcat accepted_2007_to_2018Q4.csv.gz | head -n to get columns

In [29]:
categorical = ["term","grade","sub_grade","emp_length","home_ownership","loan_status","purpose"]

numerical = ["loan_amnt","int_rate","installment","annual_inc","dti","fico_range_low",
       "fico_range_high","num_tl_90g_dpd_24m","delinq_2yrs","revol_util",
       "total_acc","open_acc","out_prncp","total_pymnt"]

In [30]:
dtypes = { **{c: 'category' for c in categorical}, **{c: 'float32' for c in numerical} }

In [31]:
df = pd.read_csv(
    "Data/accepted_2007_to_2018Q4.csv.gz",
    usecols=categorical + numerical + ['issue_d'],
    dtype=dtypes,
    parse_dates=['issue_d'],
    date_format='%b-%Y',
)

df.to_parquet('Data/accepted.parquet')

In [70]:
df = pd.read_parquet("Data/accepted.parquet")

In [42]:
db = duckdb.sql

In [56]:
db("""
SELECT COUNT(*) AS total_rows
FROM 'Data/accepted.parquet';
""").df()

,total_rows
0,2260701


In [44]:
db("""
SELECT *
FROM 'Data/accepted.parquet'
LIMIT 5;
""").df()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,issue_d,...,dti,delinq_2yrs,fico_range_low,fico_range_high,open_acc,revol_util,total_acc,out_prncp,total_pymnt,num_tl_90g_dpd_24m
0,3600.0,36 months,13.990000,123.029999,C,C4,10+ years,MORTGAGE,55000.0,2015-12-01,...,5.910000,0.0,675.0,679.0,7.0,29.700001,13.0,0.000000,4421.724121,0.0
1,24700.0,36 months,11.990000,820.280029,C,C1,10+ years,MORTGAGE,65000.0,2015-12-01,...,16.059999,1.0,715.0,719.0,22.0,19.200001,38.0,0.000000,25679.660156,0.0
2,20000.0,60 months,10.780000,432.660004,B,B4,10+ years,MORTGAGE,63000.0,2015-12-01,...,10.780000,0.0,695.0,699.0,6.0,56.200001,18.0,0.000000,22705.923828,0.0
3,35000.0,60 months,14.850000,829.900024,C,C5,10+ years,MORTGAGE,110000.0,2015-12-01,...,17.059999,0.0,785.0,789.0,13.0,11.600000,17.0,15897.650391,31464.009766,0.0
4,10400.0,60 months,22.450001,289.910004,F,F1,3 years,MORTGAGE,104433.0,2015-12-01,...,25.370001,1.0,695.0,699.0,12.0,64.500000,35.0,0.000000,11740.500000,0.0


In [55]:
db("""
DESCRIBE 'Data/accepted.parquet';
""").df()

,column_name,column_type,null,key,default,extra
0,loan_amnt,FLOAT,YES,None,None,None
1,term,VARCHAR,YES,None,None,None
2,int_rate,FLOAT,YES,None,None,None
3,installment,FLOAT,YES,None,None,None
4,grade,VARCHAR,YES,None,None,None
5,sub_grade,VARCHAR,YES,None,None,None
6,emp_length,VARCHAR,YES,None,None,None
7,home_ownership,VARCHAR,YES,None,None,None
8,annual_inc,FLOAT,YES,None,None,None
9,issue_d,TIMESTAMP,YES,None,None,None


In [75]:
columns = db("""
DESCRIBE 'Data/accepted.parquet';
""").df()['column_name'].tolist()

null = ", ".join([
    f"SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS null_{col}" 
    for col in columns
])

query = f"""
SELECT COUNT(*) AS Total_Rows,
{null}
from 'Data/accepted.parquet';
"""

db(query).df().T


,0
Total_Rows,2260701.0
null_loan_amnt,33.0
null_term,33.0
null_int_rate,33.0
null_installment,33.0
null_grade,33.0
null_sub_grade,33.0
null_emp_length,146940.0
null_home_ownership,33.0
null_annual_inc,37.0
